<a href="https://colab.research.google.com/github/demsaid400-cpu/DI-BOOTCAMP/blob/main/Exercices_XP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LoRALayer(nn.Module):
    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        std_dev = 1 / torch.sqrt(torch.tensor(rank).float())

        self.A = nn.Parameter(torch.randn(in_dim, rank) * std_dev)
        self.B = nn.Parameter(torch.zeros(rank, out_dim))
        self.alpha = alpha

    def forward(self, x):
        # x: (batch, in_dim)
        lora_update = x @ self.A @ self.B
        return self.alpha * lora_update

In [3]:
random_seed = 42
torch.manual_seed(random_seed)

x = torch.randn(2, 5)
layer = LoRALayer(in_dim=5, out_dim=3, rank=2, alpha=1.0)

print("Input x:", x)
print("Layer:", layer)
print('LoRA output:', layer(x))

Input x: tensor([[ 0.3367,  0.1288,  0.2345,  0.2303, -1.1229],
        [-0.1863,  2.2082, -0.6380,  0.4617,  0.2674]])
Layer: LoRALayer()
LoRA output: tensor([[0., 0., 0.],
        [0., 0., 0.]], grad_fn=<MulBackward0>)


In [4]:
class LinearWithLoRA(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features,
            linear.out_features,
            rank,
            alpha
        )

    def forward(self, x):
        return self.linear(x) + self.lora(x)

In [5]:
layer_base = nn.Linear(5, 3)
x = torch.randn(2, 5)

print(layer_base(x))

layer_lora_1 = LinearWithLoRA(layer_base, rank=2, alpha=1.0)
print(layer_lora_1(x))

tensor([[ 0.3312,  0.5060, -0.1719],
        [ 0.6996, -0.4606,  0.4259]], grad_fn=<AddmmBackward0>)
tensor([[ 0.3312,  0.5060, -0.1719],
        [ 0.6996, -0.4606,  0.4259]], grad_fn=<AddBackward0>)


In [9]:
class LinearWithLoRAMerged(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features,
            linear.out_features,
            rank,
            alpha
        )

    def forward(self, x):
        lora_matrix = self.lora.A @ self.lora.B  # (in, out)
        combined_weight = self.linear.weight + self.lora.alpha * lora_matrix.T
        return F.linear(x, combined_weight, self.linear.bias)

In [10]:
layer_lora_2 = LinearWithLoRAMerged(nn.Linear(5, 3), rank=2, alpha=1.0)
print(layer_lora_2(x))

tensor([[-0.7919, -0.1875, -0.2996],
        [-0.1482, -0.7191, -1.3536]], grad_fn=<AddmmBackward0>)


In [11]:
class MultilayerPerceptron(nn.Module):
    def __init__(self, num_features, num_hidden_1, num_hidden_2, num_classes):
        super().__init__()

        self.layers = nn.Sequential(
            nn.Linear(num_features, num_hidden_1),
            nn.ReLU(),
            nn.Linear(num_hidden_1, num_hidden_2),
            nn.ReLU(),
            nn.Linear(num_hidden_2, num_classes)
        )

    def forward(self, x):
        return self.layers(x)

In [12]:
num_features = 28 * 28
num_hidden_1 = 128
num_hidden_2 = 64
num_classes = 10

learning_rate = 0.001
num_epochs = 2

In [19]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

BATCH_SIZE = 64

train_dataset = datasets.MNIST(
    root='data',
    train=True,
    transform=transforms.ToTensor(),
    download=True
)

test_dataset = datasets.MNIST(
    root='data',
    train=False,
    transform=transforms.ToTensor(),
    download=True
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [15]:
def compute_accuracy(model, data_loader, device):
    model.eval()
    correct_pred, num_examples = 0, 0

    with torch.no_grad():
        for features, targets in data_loader:
            features = features.view(features.size(0), -1).to(device)
            targets = targets.to(device)

            logits = model(features)

            _, predicted_labels = torch.max(logits, 1)

            num_examples += targets.size(0)
            correct_pred += (predicted_labels == targets).sum()

    return (correct_pred.float() / num_examples) * 100

In [16]:
def train(num_epochs, model, optimizer, train_loader, device):
    start_time = time.time()

    for epoch in range(num_epochs):
        model.train()

        for batch_idx, (features, targets) in enumerate(train_loader):

            features = features.view(features.size(0), -1).to(device)
            targets = targets.to(device)

            logits = model(features)
            loss = F.cross_entropy(logits, targets)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if not batch_idx % 400:
                print(f'Epoch: {epoch+1}/{num_epochs} | Batch {batch_idx} | Loss: {loss.item():.4f}')

        acc = compute_accuracy(model, train_loader, device)
        print(f'Epoch {epoch+1} training accuracy: {acc:.2f}%')

    print(f'Total time: {(time.time()-start_time)/60:.2f} min')

In [20]:
import time

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Re-initialize and train with correct train_loader
model = MultilayerPerceptron(num_features, num_hidden_1, num_hidden_2, num_classes)
model.to(DEVICE)

optimizer_pretrained = torch.optim.Adam(model.parameters(), lr=learning_rate)

train(num_epochs, model, optimizer_pretrained, train_loader, DEVICE)
print(f'Test accuracy: {compute_accuracy(model, test_loader, DEVICE):.2f}%')

Epoch: 1/2 | Batch 0 | Loss: 2.2971
Epoch: 1/2 | Batch 400 | Loss: 0.3249
Epoch: 1/2 | Batch 800 | Loss: 0.3170
Epoch 1 training accuracy: 94.28%
Epoch: 2/2 | Batch 0 | Loss: 0.1339
Epoch: 2/2 | Batch 400 | Loss: 0.2116
Epoch: 2/2 | Batch 800 | Loss: 0.1374
Epoch 2 training accuracy: 97.08%
Total time: 0.46 min
Test accuracy: 96.57%


In [21]:
# 1. Freeze the base model parameters
for param in model.parameters():
    param.requires_grad = False

# 2. Wrap the layers with LoRA
# We'll replace the Linear layers at index 0, 2, and 4 in the Sequential block
model.layers[0] = LinearWithLoRAMerged(model.layers[0], rank=4, alpha=8.0)
model.layers[2] = LinearWithLoRAMerged(model.layers[2], rank=4, alpha=8.0)
model.layers[4] = LinearWithLoRAMerged(model.layers[4], rank=4, alpha=8.0)

model.to(DEVICE)

# 3. Check trainable parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Trainable parameters: {count_parameters(model):,}')
print(model)

Trainable parameters: 4,712
MultilayerPerceptron(
  (layers): Sequential(
    (0): LinearWithLoRAMerged(
      (linear): Linear(in_features=784, out_features=128, bias=True)
      (lora): LoRALayer()
    )
    (1): ReLU()
    (2): LinearWithLoRAMerged(
      (linear): Linear(in_features=128, out_features=64, bias=True)
      (lora): LoRALayer()
    )
    (3): ReLU()
    (4): LinearWithLoRAMerged(
      (linear): Linear(in_features=64, out_features=10, bias=True)
      (lora): LoRALayer()
    )
  )
)


In [22]:
optimizer_lora = torch.optim.Adam(model.parameters(), lr=learning_rate)

print("Starting LoRA fine-tuning...")
train(num_epochs, model, optimizer_lora, train_loader, DEVICE)

print(f'Final test accuracy after LoRA fine-tuning: {compute_accuracy(model, test_loader, DEVICE):.2f}%')

Starting LoRA fine-tuning...
Epoch: 1/2 | Batch 0 | Loss: 0.0488
Epoch: 1/2 | Batch 400 | Loss: 0.0702
Epoch: 1/2 | Batch 800 | Loss: 0.0597
Epoch 1 training accuracy: 97.13%
Epoch: 2/2 | Batch 0 | Loss: 0.0995
Epoch: 2/2 | Batch 400 | Loss: 0.0195
Epoch: 2/2 | Batch 800 | Loss: 0.0503
Epoch 2 training accuracy: 97.11%
Total time: 0.51 min
Final test accuracy after LoRA fine-tuning: 96.52%


In [25]:
import copy

# Create a fresh model instance to avoid double-wrapping
model_lora = MultilayerPerceptron(num_features, num_hidden_1, num_hidden_2, num_classes)
model_lora.to(DEVICE)

# Wrap the fresh linear layers with LoRA
model_lora.layers[0] = LinearWithLoRAMerged(model_lora.layers[0], rank=4, alpha=8)
model_lora.layers[2] = LinearWithLoRAMerged(model_lora.layers[2], rank=4, alpha=8)
model_lora.layers[4] = LinearWithLoRAMerged(model_lora.layers[4], rank=4, alpha=8)

model_lora.to(DEVICE)
print("LoRA model initialized successfully.")

LoRA model initialized successfully.


In [28]:
def freeze_linear_layers(model):
    for child in model.children():
        for layer in child:
            if isinstance(layer, LinearWithLoRAMerged):
                for param in layer.linear.parameters():
                    param.requires_grad = False

freeze_linear_layers(model_lora)

for name, param in model_lora.named_parameters():
    print(f"{name:50} | Trainable: {param.requires_grad}")

layers.0.linear.weight                             | Trainable: False
layers.0.linear.bias                               | Trainable: False
layers.0.lora.A                                    | Trainable: True
layers.0.lora.B                                    | Trainable: True
layers.2.linear.weight                             | Trainable: False
layers.2.linear.bias                               | Trainable: False
layers.2.lora.A                                    | Trainable: True
layers.2.lora.B                                    | Trainable: True
layers.4.linear.weight                             | Trainable: False
layers.4.linear.bias                               | Trainable: False
layers.4.lora.A                                    | Trainable: True
layers.4.lora.B                                    | Trainable: True


In [27]:
optimizer_lora = torch.optim.Adam(model_lora.parameters(), lr=learning_rate)

train(num_epochs, model_lora, optimizer_lora, train_loader, DEVICE)

print(compute_accuracy(model_lora, test_loader, DEVICE))

Epoch: 1/2 | Batch 0 | Loss: 2.3019
Epoch: 1/2 | Batch 400 | Loss: 0.2254
Epoch: 1/2 | Batch 800 | Loss: 0.0487
Epoch 1 training accuracy: 95.85%
Epoch: 2/2 | Batch 0 | Loss: 0.0675
Epoch: 2/2 | Batch 400 | Loss: 0.1425
Epoch: 2/2 | Batch 800 | Loss: 0.1403
Epoch 2 training accuracy: 96.39%
Total time: 0.49 min
tensor(95.5700, device='cuda:0')
